# Day 5 — Delta Lake: Architecture, Formats & Schema

Plan for about 2–2.5 hours in the notebook, plus any discussion. Work through the parts in order, or split early ingest sections and later practice sections across the class if needed.

Topics include Parquet vs Delta and the transaction log, CSV and JSON loads to Delta with `overwrite` / `append` / `mergeSchema`, partitioned writes, `DESCRIBE DETAIL` and `DESCRIBE HISTORY`, strict vs evolving schema, `replaceWhere`, SQL reads on `delta.` paths, data quality checks, skill-builder cells, and optional challenges.

Prerequisites: `%run ./02-Mount-Azure-Data-Lake`, and the usual course files `2010-summary.csv` and `json/2015-summary.json` on the storage account. In UC-oriented workspaces, prefer `_metadata.file_path` (or a literal path) over `input_file_name()` for file lineage.

Public Databricks training labs on Delta reads/writes, versioning, schema evolution, and optimizations are broadly aligned with this notebook; paths here use ABFS as in the rest of this course.

## Part A — Connect to ADLS (same as Days 1–4)

In [ ]:
%run ./02-Mount-Azure-Data-Lake

In [ ]:
BASE_PATH = base_path
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
P_JSON = DAY5 + "/delta/flight_json"
P_PART = DAY5 + "/delta/flight_partitioned"
P_STRICT = DAY5 + "/delta/schema_strict"
P_EVOLVE = DAY5 + "/delta/schema_evolve_demo"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"

for _n, _p in [
    ("DAY5 root", DAY5),
    ("basic", P_BASIC),
    ("json delta", P_JSON),
    ("partitioned", P_PART),
    ("strict", P_STRICT),
    ("evolve", P_EVOLVE),
]:
    print(_n + ":", _p)

## Part B — Theory: Parquet-only lakes vs Delta

**Plain Parquet directories**
- No single **transaction log** → readers may see **partial writes** if a job fails mid-write.
- **No built-in ACID** semantics across concurrent writers.
- **Schema drift** is painful: files disagree silently.

**Delta Lake**
- **`_delta_log`** records commits (JSON + checkpoints).
- **ACID** table semantics at the directory level.
- **Time travel**, **CLONE**, **MERGE/UPDATE/DELETE**, **Change Data Feed** (later topics / Day 6).
- **Schema enforcement** and controlled **evolution**.

### B2 — Checklist (Delta features you will touch today)

| Feature | Where in Day 5 |
|---------|----------------|
| ACID overwrite / append | Parts C–E |
| Transaction log (history) | Part F |
| Schema enforcement | Part G |
| Schema evolution (`mergeSchema`) | Part G |
| Partitioning | Part E |
| Optimize (Databricks) | Notebook **03** |

### B3 — Inside `_delta_log` (mental model)

Each commit appends **JSON** files (and periodic **checkpoints**) under **`_delta_log/`** at the table root. Readers always see a **consistent snapshot** from the latest successful commit.

| Artifact | Role |
|----------|------|
| `*.json` commit | Add/remove files, schema, metadata, metrics |
| Checkpoints | Speed up log replay for large tables |
| Reader protocol | Uses log to decide which Parquet files are **active** |

**Takeaway:** Delta is **Parquet files + transaction log**, not “magic storage” — governance still lives in **ABFS ACLs** + **Unity Catalog** when you register tables.

## Part C — First Delta table: CSV → Delta (`overwrite`)

In [ ]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit

df0 = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
)
df0.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_BASIC)
print("Rows:", df0.count(), "| Path:", P_BASIC)

In [ ]:
spark.read.format("delta").load(P_BASIC).printSchema()
spark.read.format("delta").load(P_BASIC).show(5, truncate=False)

## Part D — Append with `append` + extra column (`mergeSchema`)

In [ ]:
from pyspark.sql.functions import coalesce, col, current_timestamp, lit, expr

append_batch = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_path", coalesce(col("_metadata.file_path"), lit(SOURCE_CSV)))
    .withColumn("batch_id", lit("batch_002"))
    .withColumn("load_comment", lit("append_with_mergeSchema"))
)
append_batch.write.format("delta").mode("append").option("mergeSchema", "true").save(P_BASIC)
print("After append, total rows:", spark.read.format("delta").load(P_BASIC).count())
spark.read.format("delta").load(P_BASIC).printSchema()

## Part E — JSON → Delta + **partitioned** write (lab *08* style)

In [ ]:
from pyspark.sql.functions import col, substring, upper as u

jdf = spark.read.format("json").option("inferSchema", True).load(SOURCE_JSON)
jdf = jdf.withColumn("dest_initial", u(substring(col("DEST_COUNTRY_NAME"), 1, 1)))
jdf.write.format("delta").mode("overwrite").partitionBy("dest_initial").option("overwriteSchema", "true").save(P_PART)
print("Partitioned Delta at:", P_PART)

In [ ]:
spark.read.format("delta").load(P_PART).groupBy("dest_initial").count().orderBy("dest_initial").show(30)

In [ ]:
# Read only one partition (predicate pushdown)
spark.read.format("delta").load(P_PART).where("dest_initial = 'U'").select("DEST_COUNTRY_NAME", "count").show(10, truncate=False)

## Part F — `DESCRIBE DETAIL` + `DESCRIBE HISTORY`

In [ ]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").show(truncate=False)

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").select("version", "timestamp", "operation", "operationMetrics").show(20, truncate=False)

## Part F2 — SQL on Delta path (Spark SQL `delta.\`path\``)

Same physical table as **`spark.read.format('delta').load(P_BASIC)`** — analysts often use **`delta.\`abfss://...\``** in SQL warehouses.

### F2a — Row count

In [ ]:
spark.sql(f"SELECT COUNT(*) AS c FROM delta.`{P_BASIC}`").show(truncate=False)

### F2b — Top destinations

In [ ]:
spark.sql(f"SELECT DEST_COUNTRY_NAME, SUM(count) AS s FROM delta.`{P_BASIC}` GROUP BY 1 ORDER BY 2 DESC LIMIT 8").show(truncate=False)

### F2c — Min / max count

In [ ]:
spark.sql(f"SELECT MIN(count) AS mn, MAX(count) AS mx, AVG(count) AS av FROM delta.`{P_BASIC}`").show(truncate=False)

### F2d — Distinct origins

In [ ]:
spark.sql(f"SELECT COUNT(DISTINCT ORIGIN_COUNTRY_NAME) AS origins FROM delta.`{P_BASIC}`").show(truncate=False)

### F2e — Filter + limit

In [ ]:
spark.sql(f"SELECT * FROM delta.`{P_BASIC}` WHERE count > 1000 ORDER BY count DESC LIMIT 10").show(truncate=False)

## Part G — Schema **strict** vs **evolution**

In [ ]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

strict_schema = StructType(
    [
        StructField("DEST_COUNTRY_NAME", StringType(), True),
        StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
        StructField("count", LongType(), True),
    ]
)
sdf = spark.read.format("csv").option("header", True).schema(strict_schema).load(SOURCE_CSV)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_STRICT)
print("Strict-schema Delta:", P_STRICT)

In [ ]:
# Try to append a row with an EXTRA column — expect failure without mergeSchema
from pyspark.sql import Row

bad = spark.createDataFrame([Row(DEST_COUNTRY_NAME="X", ORIGIN_COUNTRY_NAME="Y", count=1, extra_col="oops")])
try:
    bad.write.format("delta").mode("append").save(P_STRICT)
except Exception as e:
    print("Expected failure (schema mismatch):", type(e).__name__)

In [ ]:
# Same append with mergeSchema = true
bad.write.format("delta").mode("append").option("mergeSchema", "true").save(P_STRICT)
spark.read.format("delta").load(P_STRICT).printSchema()

## Part H — Multiple **read patterns** (practice)

### H1 — `format('delta').load`

In [ ]:
spark.read.format('delta').load(P_BASIC).select('DEST_COUNTRY_NAME', 'count').limit(5).show()

### H2 — SQL `delta.\`path\``

In [ ]:
spark.sql(f'SELECT COUNT(*) AS c FROM delta.`{P_BASIC}`').show()

### H3 — `table` after temp view

In [ ]:
spark.read.format('delta').load(P_BASIC).createOrReplaceTempView('t_basic'); spark.sql('SELECT MIN(count), MAX(count) FROM t_basic').show()

### H4 — JSON copy for merge lab later

In [ ]:
spark.read.format('json').load(SOURCE_JSON).write.format('delta').mode('overwrite').option('overwriteSchema','true').save(P_JSON); print('OK', P_JSON)

## Part I — `replaceWhere` (conditional overwrite) — discussion + pattern

**Idea:** Overwrite **only** rows matching a predicate (partition pruning). Reduces blast radius vs full `overwrite`.

```python
# Example pattern (uncomment & adapt partition column):
# df.write.format("delta").mode("overwrite").option("replaceWhere", "dest_initial = 'Z'").save(P_PART)
```

Use when you partition by a stable key (date, region). **Wrong predicates** can corrupt data — test in dev first.

### I2 — Hands-on: **`replaceWhere`** on a **single** partition (`P_PART`)

Run **after** Part E. This **overwrites only** rows where `dest_initial = 'U'`. We add a marker column with **`mergeSchema`** so you can see the effect in `DESCRIBE HISTORY`.

In [ ]:
from pyspark.sql.functions import col, current_timestamp, lit

slice_df = (
    spark.read.format("delta")
    .load(P_PART)
    .where("dest_initial = 'U'")
    .withColumn("partition_refresh_at", current_timestamp())
    .withColumn("replacewhere_note", lit("slice_U_refreshed"))
)
(
    slice_df.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", "dest_initial = 'U'")
    .option("mergeSchema", "true")
    .save(P_PART)
)
print("Rows in U slice after replace:", spark.read.format("delta").load(P_PART).where("dest_initial = 'U'").count())
spark.sql(f"DESCRIBE HISTORY delta.`{P_PART}`").select("version", "operation").show(5, truncate=False)

## Part K — Analytics on **`P_BASIC`** (aggregations & distribution)

In [ ]:
from pyspark.sql.functions import col

b = spark.read.format("delta").load(P_BASIC)
b.groupBy("DEST_COUNTRY_NAME").agg({"count": "sum"}).orderBy(col("sum(count)").desc()).show(12, truncate=False)
b.selectExpr("percentile_approx(count, 0.5) as median_cnt", "max(count) as max_cnt", "min(count) as min_cnt").show()

In [ ]:
from pyspark.sql.functions import col

# Top origin countries by total flights
spark.read.format("delta").load(P_BASIC).groupBy("ORIGIN_COUNTRY_NAME").sum("count").orderBy(col("sum(count)").desc()).show(12, truncate=False)

## Part L — **`P_JSON`** lifecycle + **schema** vs CSV table

In [ ]:
# Ensure P_JSON exists (idempotent)
spark.read.format("json").option("inferSchema", True).load(SOURCE_JSON).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_JSON)
spark.read.format("delta").load(P_JSON).printSchema()
spark.read.format("delta").load(P_JSON).show(5, truncate=False)

In [ ]:
# Column overlap: JSON vs basic Delta (names only)
cols_j = set(spark.read.format("delta").load(P_JSON).columns)
cols_b = set(spark.read.format("delta").load(P_BASIC).columns)
print("Only in JSON delta:", sorted(cols_j - cols_b))
print("Only in basic delta:", sorted(cols_b - cols_j))
print("Shared:", sorted(cols_j & cols_b))

## Part M — **Incremental** mindset: watermark column (`ingested_at`)

Production loads often track **last successful timestamp** or **batch id**. Here, filter **newer** rows (demo pattern only):

```python
# last_ts = spark.read.format("delta").load(P_BASIC).selectExpr("max(ingested_at)").collect()[0][0]
# incremental = spark.read.format("csv").options(header=True, inferSchema=True).load(SOURCE_CSV).where(col("ingested_at") > lit(last_ts))  # CSV has no ts — real pipelines add file date
```

Your **`ingested_at`** on `P_BASIC` is **load time**, not event time — use it to teach **metadata lineage**, not business cutoffs.

In [ ]:
from pyspark.sql.functions import col

mx = spark.read.format("delta").load(P_BASIC).selectExpr("max(ingested_at) as mx").collect()[0]["mx"]
print("Latest ingested_at in P_BASIC:", mx)
spark.read.format("delta").load(P_BASIC).where(col("batch_id").isNotNull()).select("batch_id", "ingested_at", "DEST_COUNTRY_NAME").show(8, truncate=False)

## Part N — **Quality** checks: nulls & duplicate route keys

In [ ]:
from pyspark.sql.functions import col, count

b = spark.read.format("delta").load(P_BASIC)
b.select([count(col(c)).alias(c) for c in ["DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count"]]).show()
b.groupBy("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME").count().where("count > 1").orderBy(col("count").desc()).show(10, truncate=False)

## Part O — **Table properties** & **detail** for multiple tables

In [ ]:
for label, path in [("basic", P_BASIC), ("partitioned", P_PART), ("json", P_JSON)]:
    print("===", label, "===")
    spark.sql(f"DESCRIBE DETAIL delta.`{path}`").select("format", "partitionColumns", "numFiles").show(truncate=False)

In [ ]:
spark.sql(f"SHOW TBLPROPERTIES delta.`{P_BASIC}`").show(30, truncate=False)

## Part P — **Worked skill builders** (run each cell; tweak filters)

### P1 — Countries with `count` above global average

In [ ]:
from pyspark.sql.functions import avg, col

b = spark.read.format("delta").load(P_BASIC)
avg_c = b.select(avg("count").alias("a")).collect()[0]["a"]
b.where(col("count") > avg_c).select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count").orderBy(col("count").desc()).show(15, truncate=False)
print("avg count:", avg_c)

### P2 — Create temp view + SQL window (rank by destination)

In [ ]:
spark.read.format("delta").load(P_BASIC).createOrReplaceTempView("flight_basic")
spark.sql(
    '''
  SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count,
         ROW_NUMBER() OVER (PARTITION BY DEST_COUNTRY_NAME ORDER BY count DESC) AS rnk
  FROM flight_basic
'''
).where("rnk <= 3").show(20, truncate=False)

### P3 — Read **one** partition + cache pattern

In [ ]:
from pyspark.sql.functions import col

ud = spark.read.format("delta").load(P_PART).where("dest_initial = 'U'")
print("partition rows:", ud.count())
ud.groupBy("DEST_COUNTRY_NAME").sum("count").orderBy(col("sum(count)").desc()).show(10, truncate=False)

### P4 — Union CSV-shaped slice with JSON delta (align columns)

In [ ]:
from pyspark.sql.functions import lit

a = spark.read.format("delta").load(P_BASIC).select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count").limit(50).withColumn("src", lit("basic"))
j = spark.read.format("delta").load(P_JSON).select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count").limit(50).withColumn("src", lit("json"))
a.unionByName(j).groupBy("src").count().show()

### P5 — `coalesce` for display (nullable batch metadata)

In [ ]:
from pyspark.sql.functions import coalesce, col, lit

spark.read.format("delta").load(P_BASIC).select(
    "DEST_COUNTRY_NAME", coalesce(col("batch_id"), lit("initial_load")).alias("batch_label")
).show(12, truncate=False)

### P6 — `overwrite` the **strict** table with a **smaller** schema file (expect clean load)

In [ ]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

small = StructType(
    [
        StructField("DEST_COUNTRY_NAME", StringType(), True),
        StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
        StructField("count", LongType(), True),
    ]
)
spark.read.format("csv").option("header", True).schema(small).load(SOURCE_CSV).limit(100).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_STRICT)
print("rows", spark.read.format("delta").load(P_STRICT).count())

### P7 — Compare **row counts** `P_BASIC` vs re-read from **same path** (consistency)

In [ ]:
c1 = spark.read.format("delta").load(P_BASIC).count()
c2 = spark.sql(f"SELECT COUNT(*) AS c FROM delta.`{P_BASIC}`").collect()[0]["c"]
print(c1, c2, "match" if c1 == c2 else "MISMATCH")

### P8 — `input_file_name()` **avoid** in UC; use `_metadata` (already in pipeline)

In [ ]:
# UC-safe: _metadata.file_path was stored in source_path on ingest
spark.read.format("delta").load(P_BASIC).select("source_path").distinct().show(5, truncate=False)

### P9 — `mergeSchema` vs `overwriteSchema` (scratch table — does not touch P_BASIC)

In [ ]:
# mergeSchema: widen schema on APPEND (additive)
# overwriteSchema: replace schema on OVERWRITE (breaking — use in dev with backups)
from pyspark.sql.functions import lit

P9 = DAY5 + "/delta/p9_schema_demo"
spark.read.format("delta").load(P_BASIC).limit(50).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P9)
spark.read.format("delta").load(P9).limit(5).withColumn("demo_tag", lit("p9")).write.format("delta").mode("append").option("mergeSchema", "true").save(P9)
print("Scratch table schema after widen-append:")
spark.read.format("delta").load(P9).printSchema()

### P10 — `EXPLAIN` on Delta read (predicate pushdown)

In [ ]:
spark.sql(f"EXPLAIN EXTENDED SELECT * FROM delta.`{P_BASIC}` WHERE DEST_COUNTRY_NAME = 'Germany'").show(truncate=False)

### P11 — Partition pruning: count rows without scanning whole table (partitioned path)

In [ ]:
# Should prune to one partition when dest_initial is partition column
spark.read.format("delta").load(P_PART).where("dest_initial = 'A'").groupBy().count().show()
spark.sql(f"EXPLAIN SELECT COUNT(*) FROM delta.`{P_PART}` WHERE dest_initial = 'A'").show(truncate=False)

### P12 — Compare **`P_JSON`** vs **`P_BASIC`** row counts + sample join on route key

In [ ]:
from pyspark.sql.functions import col

jb = spark.read.format("delta").load(P_JSON).alias("j")
bb = spark.read.format("delta").load(P_BASIC).alias("b")
joined = jb.join(bb, on=["DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME"], how="inner")
print("JSON rows:", spark.read.format("delta").load(P_JSON).count())
print("Basic rows:", spark.read.format("delta").load(P_BASIC).count())
print("Inner join rows (sample overlap):", joined.count())
joined.select("j.DEST_COUNTRY_NAME", "j.count", "b.count").limit(8).show(truncate=False)

## Part J — Mini challenges

Complete in the cells below. Compare with others in the class or review answers when the group reconvenes.

### J1 — Write **`DAY5 + '/delta/challenge_distinct'`** as Delta: distinct **`DEST_COUNTRY_NAME`** from `P_BASIC` with a column **`first_seen`** = current_timestamp().

In [ ]:
# Your solution:


### J2 — From **`P_PART`**, compute **top 5** `dest_initial` by total **`sum(count)`**.

In [ ]:
# Your solution:


### J3 — Using **`DESCRIBE HISTORY`**, identify the **latest** operation on **`P_BASIC`** and read **`operationMetrics`**.

In [ ]:
# Your solution:


### J4 — Write SQL only: **`SELECT`** from **`delta.\`P_STRICT\``** (use f-string) — count rows where **`count` > 500**.

In [ ]:
# Your solution:


### J5 — Re-read **`P_BASIC`** with **`option('versionAsOf', 0)`** (if version 0 exists). Compare row count to latest — when do they differ?

In [ ]:
# Your solution:


### J6 — One paragraph: **Why is `replaceWhere` safer than full `overwrite` on a partitioned table?** When is it still dangerous?

In [ ]:
# Your solution:


## Part Q — **Failure** story: bad append + recovery (`mergeSchema` / `overwrite`)

In [ ]:
from pyspark.sql import Row

# Try to append a row with wrong type for count — expect failure without casting
bad_type = spark.createDataFrame([Row(DEST_COUNTRY_NAME="ZZ", ORIGIN_COUNTRY_NAME="YY", count="not_a_number")])
try:
    bad_type.write.format("delta").mode("append").save(P_STRICT)
except Exception as e:
    print("Expected type / parse issue:", type(e).__name__)
    print(str(e)[:200])

## Next notebooks (Day 5)

1. **`03-Day5-Delta-History-Optimize-Advanced.ipynb`** — time travel, restore concepts, **OPTIMIZE**, **ZORDER**, **VACUUM**, CDF intro.
2. **`04-Day5-Delta-DML-MERGE-SCD.ipynb`** — **MERGE**, **UPDATE**, **DELETE**, **SCD Type 1 / 2** patterns.

---

### Cleanup (optional)

```python
# from pyspark.sql import SparkSession
# dbutils.fs.rm(DAY5.replace("abfss://", "/dbfs/..."), True)  # only if you use DBFS — prefer Azure portal / lifecycle policy for abfss
```